# 03 — Stats and plots

Paired comparisons, mixed-effects model with `t*condition` interaction, effect sizes (Cohen's d_z), and the headline presentation figures.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', context='talk')

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data') if os.path.basename(os.getcwd()) == 'src' else 'data'
FIG_DIR = os.path.join(os.path.dirname(os.getcwd()), 'figs') if os.path.basename(os.getcwd()) == 'src' else 'figs'
os.makedirs(FIG_DIR, exist_ok=True)
pf = pd.read_parquet(os.path.join(DATA_DIR, 'revision_pairs_features.parquet'))
print(pf.shape)
print(pf['condition'].value_counts().to_dict())

(267, 41)
{'stairs': 157, 'random_reread': 110}


## Paired tests and Cohen's d_z by feature and condition

In [2]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) + '/src' if os.path.basename(os.getcwd())!='src' else '.')
import features as F

FEATURES = F.FEATURE_COLUMNS + [
    'content_score','language_score','similarity_score','containment_score',
]

def cohen_dz(deltas):
    d = np.asarray(deltas, dtype=float)
    d = d[~np.isnan(d)]
    if len(d) < 2 or d.std(ddof=1) == 0:
        return np.nan
    return d.mean() / d.std(ddof=1)

rows = []
for feat in FEATURES:
    dcol = f'{feat}_delta'
    if dcol not in pf.columns:
        continue
    for cond in ['stairs','random_reread']:
        sub = pf[pf['condition']==cond][dcol].dropna()
        if len(sub) < 2:
            continue
        t_stat, p_t = stats.ttest_1samp(sub, 0.0)
        try:
            w_stat, p_w = stats.wilcoxon(sub, zero_method='wilcox')
        except ValueError:
            w_stat, p_w = (np.nan, np.nan)
        rows.append({
            'feature': feat,
            'condition': cond,
            'n': len(sub),
            'mean_delta': sub.mean(),
            'sd_delta': sub.std(ddof=1),
            'cohens_dz': cohen_dz(sub),
            't': t_stat,
            'p_t': p_t,
            'wilcoxon_W': w_stat,
            'p_wilcoxon': p_w,
        })
paired_stats = pd.DataFrame(rows)
paired_stats.to_csv(os.path.join(DATA_DIR, 'paired_stats.csv'), index=False)
paired_stats

/opt/conda/lib/python3.11/site-packages/scipy/stats/_morestats.py:4102: UserWarning: Sample size too small for normal approximation.
  warnings.warn("Sample size too small for normal approximation.")


,feature,condition,n,mean_delta,sd_delta,cohens_dz,t,p_t,wilcoxon_W,p_wilcoxon
0,word_count,stairs,157,23.936306,25.676159,0.932239,11.680916,4.781327e-23,276.0,2.226737e-24
1,word_count,random_reread,110,5.645455,16.678056,0.338496,3.550176,5.696327e-04,0.0,2.930525e-04
2,sentence_count,stairs,157,1.165605,1.399850,0.832665,10.433257,1.156816e-19,556.0,1.922231e-19
3,sentence_count,random_reread,110,0.300000,0.904119,0.331815,3.480102,7.218218e-04,0.0,5.760403e-04
4,mtld,stairs,157,2.137253,18.082267,0.118196,1.480992,1.406256e-01,4854.0,5.902731e-02
5,mtld,random_reread,110,1.911061,11.814408,0.161757,1.696520,9.264159e-02,42.0,1.024344e-01
6,critical_thinking,stairs,157,0.032217,0.480028,0.067116,0.840955,4.016602e-01,277.0,3.789645e-01
7,critical_thinking,random_reread,110,0.003430,0.154769,0.022163,0.232449,8.166251e-01,5.0,1.000000e+00
8,confusion,stairs,157,-0.014811,0.338602,-0.043742,-0.548084,5.844180e-01,50.0,5.700609e-01
9,confusion,random_reread,110,-0.001026,0.010764,-0.095346,-1.000000,3.195253e-01,0.0,3.173105e-01


## Mixed-effects model: feature ~ t * condition + (1|user) + (1|page)

In [3]:
# Reshape to long form (one row per observation, with a t column).
long_rows = []
for _, r in pf.iterrows():
    base = {'user_id': r['user_id'], 'page_slug': r['page_slug'], 'condition': r['condition']}
    for t_val, suffix in [(1,'t1'),(2,'t2')]:
        obs = dict(base, t=t_val)
        for feat in FEATURES:
            col = f'{feat}_{suffix}'
            if col in pf.columns:
                obs[feat] = r[col]
        long_rows.append(obs)
long_df = pd.DataFrame(long_rows)
long_df['condition'] = pd.Categorical(long_df['condition'], categories=['random_reread','stairs'])
long_df.to_csv(os.path.join(DATA_DIR, 'revision_pairs_long.csv'), index=False)
print(long_df.shape)
long_df.head()

(534, 13)


,user_id,page_slug,condition,t,word_count,sentence_count,mtld,critical_thinking,confusion,content_score,language_score,similarity_score,containment_score
0,2hrsmtdsvpnutaqxmgdnaaldvq,7-1-the-relatively-recent-arrival-of-economic-...,stairs,1,70.0,2.0,78.415556,0.0,0.0,-0.797054,3.142005,0.662562,0.0000
1,2hrsmtdsvpnutaqxmgdnaaldvq,7-1-the-relatively-recent-arrival-of-economic-...,stairs,2,100.0,3.0,60.293224,0.0,0.0,-0.236283,3.231656,0.658375,0.0435
2,2hrsmtdsvpnutaqxmgdnaaldvq,7-2-labor-productivity-and-economic-growth,stairs,1,56.0,3.0,57.079964,0.0,0.0,-0.093934,2.612348,0.715804,0.0000
3,2hrsmtdsvpnutaqxmgdnaaldvq,7-2-labor-productivity-and-economic-growth,stairs,2,97.0,6.0,55.435243,0.0,0.0,0.578476,2.680135,0.852832,0.0000
4,2qmmqqwwa5a7skhydrbq5u67ku,7-1-the-relatively-recent-arrival-of-economic-...,stairs,1,59.0,4.0,88.607273,0.0,0.0,-0.779259,2.618597,0.689911,0.0556


In [4]:
# statsmodels MixedLM supports only a single grouping factor; we use user_id.
# This mirrors the LME pattern in itell-data/2024.05-CTTC/src/lme/.
lme_rows = []
for feat in FEATURES:
    d = long_df[['user_id','page_slug','condition','t', feat]].dropna()
    if d[feat].nunique() < 3:
        continue
    try:
        md = smf.mixedlm(f'{feat} ~ t * C(condition, Treatment("random_reread"))',
                         data=d, groups=d['user_id'])
        mdf = md.fit(method='lbfgs', reml=True, disp=False)
    except Exception as e:
        print('skip', feat, e)
        continue
    # interaction term is the row whose name contains both 't' and 'condition'.
    interaction_key = [k for k in mdf.params.index if ':' in k and 't' in k]
    if not interaction_key:
        continue
    k = interaction_key[0]
    lme_rows.append({
        'feature': feat,
        'interaction_term': k,
        'coef': mdf.params[k],
        'se': mdf.bse[k],
        'z': mdf.tvalues[k],
        'p': mdf.pvalues[k],
    })
lme_summary = pd.DataFrame(lme_rows)
lme_summary.to_csv(os.path.join(DATA_DIR, 'lme_interactions.csv'), index=False)
lme_summary

/opt/conda/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/conda/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/conda/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


skip mtld Singular matrix
skip critical_thinking Singular matrix
skip confusion Singular matrix


/opt/conda/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


skip similarity_score Singular matrix


,feature,interaction_term,coef,se,z,p
0,word_count,"t:C(condition, Treatment(""random_reread""))[T.s...",18.290851,3.394191,5.388869,7.090248e-08
1,sentence_count,"t:C(condition, Treatment(""random_reread""))[T.s...",0.865605,0.207778,4.166013,3.099728e-05
2,content_score,"t:C(condition, Treatment(""random_reread""))[T.s...",0.276473,0.063540,4.351191,1.354003e-05
3,language_score,"t:C(condition, Treatment(""random_reread""))[T.s...",0.056658,0.043739,1.295370,1.951925e-01
4,containment_score,"t:C(condition, Treatment(""random_reread""))[T.s...",0.007750,0.019853,0.390393,6.962457e-01


## Headline plot: Cohen's d_z by feature and condition

Effect sizes are used so heterogeneous features (counts, rates, existing iTELL scores) can be read on a common axis.

In [ ]:
HEADLINE = ['word_count','sentence_count','mtld','content_score',
            'similarity_score','containment_score',
            'critical_thinking','confusion']
LABELS = {
    'word_count': 'word count',
    'sentence_count': 'sentence count',
    'mtld': 'MTLD',
    'content_score': 'content',
    'similarity_score': 'relevance',
    'containment_score': 'borrowing',
    'critical_thinking': 'cognition',
    'confusion': 'uncertainty',
}
label_order = [LABELS[f] for f in HEADLINE]

plot_df = (paired_stats[paired_stats['feature'].isin(HEADLINE)]
           .assign(feature=lambda d: pd.Categorical(d['feature'], categories=HEADLINE, ordered=True))
           .sort_values(['feature','condition']))
plot_df['label'] = pd.Categorical(plot_df['feature'].map(LABELS),
                                   categories=label_order, ordered=True)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(plot_df, x='label', y='cohens_dz', hue='condition', ax=ax,
            palette={'stairs':'#2b7a78','random_reread':'#c0c0c0'},
            hue_order=['stairs','random_reread'])
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel("Cohen's d$_z$  (paired t1 → t2)")
ax.set_xlabel('')
ax.set_title('Revision-driven change: STAIRS vs. control')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'headline_effect_sizes.png'), dpi=180)
plt.show()

In [ ]:
# Effect-size heatmap across the headline features
piv = (paired_stats[paired_stats['feature'].isin(HEADLINE)]
       .pivot(index='feature', columns='condition', values='cohens_dz')
       .reindex(HEADLINE)
       .rename(index=LABELS))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(piv, annot=True, fmt='.2f', center=0, cmap='RdBu_r', ax=ax, cbar_kws={'label':"Cohen's dz"})
ax.set_title('Paired effect sizes (t1 → t2)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'effect_size_heatmap.png'), dpi=180)
plt.show()

## Qualitative examples — feedback → before → after

In [7]:
stairs_pairs = pf[(pf['condition']=='stairs') & (pf['stairs_feedback'].str.len()>0)].copy()
# Rank by magnitude on a few dimensions of interest.
stairs_pairs['score_combo'] = (stairs_pairs['content_score_delta'].fillna(0).rank()
                                + stairs_pairs['critical_thinking_delta'].fillna(0).rank()
                                + stairs_pairs['word_count_delta'].fillna(0).rank())
examples = stairs_pairs.sort_values('score_combo', ascending=False).head(3)

for _, r in examples.iterrows():
    print('='*80)
    print(f"user={r['user_id'][:8]}  page={r['page_slug']}  "
          f"Δcontent={r['content_score_delta']:+.2f}  "
          f"Δcrit={r['critical_thinking_delta']:+.2f}  "
          f"Δwc={int(r['word_count_delta']):+d}")
    print('--- STAIRS feedback ---')
    print(r['stairs_feedback'][:800])
    print('--- BEFORE (t1) ---')
    print(r['text_t1'])
    print('--- AFTER (t2) ---')
    print(r['text_t2'])

examples[['user_id','page_slug','content_score_delta','critical_thinking_delta',
          'word_count_delta','levenshtein']].to_csv(
    os.path.join(DATA_DIR, 'qualitative_examples.csv'), index=False)

user=tpe6f6ja  page=7-1-the-relatively-recent-arrival-of-economic-growth  Δcontent=+0.96  Δcrit=+0.61  Δwc=+95
--- STAIRS feedback ---


How do the concepts discussed in this excerpt relate to the idea that economic growth is fostered by strong legal systems and property rights, as mentioned in the user's summary?
---


That's correct! The excerpt explains how adherence to the rule of law and protection of property rights by a country's government enables markets to work effectively and efficiently, leading to economic growth. It also highlights the importance of clear, public, fair, enforced, and equally applicable laws to ensure that individuals and firms can enter into contracts and use their property as they see fit.
--- BEFORE (t1) ---
Economic growth depends on;

Rule of law
 Property rights
Contractual rights
Clear laws
Independent courts

Countries with strong rights and laws have effective legal systems and fosters economic growth while on other hand 
Countries with weak right